In [0]:
print("Retail Data Engineering Training")
print("Environment check successful!")

print("Python version:", __import__("sys").version)
print("Spark version:", spark.version)

Retail Data Engineering Training
Environment check successful!
Python version: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Spark version: 4.2.0


In [0]:
print("Spark version:", spark.version)

print("\nAvailable catalogs:")
display(spark.sql("SHOW CATALOGS"))

Spark version: 4.2.0

Available catalogs:


catalog
formula1
formula1_incr
samples
system
workspace


In [0]:
catalogs = spark.sql("SHOW CATALOGS")
display(catalogs)

catalog
formula1
formula1_incr
samples
system
workspace


In [0]:
catalogs = spark.sql("SHOW CATALOGS").collect()

for catalog in catalogs:
    print(catalog[0])

formula1
formula1_incr
samples
system
workspace


In [0]:
print("Current catalog and schema:")
display(spark.sql("SELECT current_catalog(), current_schema()"))

print("Schemas in the workspace catalog:")
display(spark.sql("SHOW SCHEMAS IN workspace"))

Current catalog and schema:


current_catalog(),current_schema()
workspace,default


Schemas in the workspace catalog:


databaseName
default
information_schema


In [0]:
%sql
SHOW GRANTS ON CATALOG workspace;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.retail_training;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
retail_training


In [0]:
%sql
DESCRIBE SCHEMA EXTENDED workspace.retail_training;

database_description_item,database_description_value
Catalog Name,workspace
Namespace Name,retail_training
Comment,
Location,
Owner,paslacerna@gmail.com
Properties,"((unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/retail_training/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/retail_training/raw_files/customers.csv,customers.csv,1197,1792080912000
dbfs:/Volumes/workspace/retail_training/raw_files/products.csv,products.csv,375,1792080911000
dbfs:/Volumes/workspace/retail_training/raw_files/sales.csv,sales.csv,2934,1792080912000
dbfs:/Volumes/workspace/retail_training/raw_files/stores.csv,stores.csv,147,1792080912000


In [0]:

from pyspark.sql import functions as F

# Source folder
raw_path = "/Volumes/workspace/retail_training/raw_files"

# Destination schema
target_schema = "workspace.retail_training"

# CSV files to load
tables = ["customers", "products", "stores", "sales"]

for table_name in tables:
    file_path = f"{raw_path}/{table_name}.csv"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    # Add ingestion metadata
    df = df.withColumn("_ingested_at", F.current_timestamp())

    # Save as a Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{target_schema}.bronze_{table_name}")
    )

    print(f"Loaded bronze_{table_name}: {df.count()} rows")


Loaded bronze_customers: 20 rows
Loaded bronze_products: 12 rows
Loaded bronze_stores: 5 rows
Loaded bronze_sales: 100 rows


In [0]:
%sql
SHOW TABLES IN workspace.retail_training;

database,tableName,isTemporary
retail_training,bronze_customers,false
retail_training,bronze_products,false
retail_training,bronze_sales,false
retail_training,bronze_stores,false


In [0]:
%sql
SELECT
    sale_id,
    customer_id,
    product_id,
    quantity,
    unit_price,
    sale_date,
    _ingested_at
FROM workspace.retail_training.bronze_sales
ORDER BY sale_id
LIMIT 10;

sale_id,customer_id,product_id,quantity,unit_price,sale_date,_ingested_at
1,1,11,1,16000,2026-02-01,2026-10-15T16:20:02.150Z
2,4,4,2,9800,2026-01-12,2026-10-15T16:20:02.150Z
3,2,10,4,7200,2026-01-12,2026-10-15T16:20:02.150Z
4,17,4,2,9800,2026-01-04,2026-10-15T16:20:02.150Z
5,18,9,2,1800,2026-01-29,2026-10-15T16:20:02.150Z
6,1,8,3,28000,2026-03-31,2026-10-15T16:20:02.150Z
7,9,7,3,3200,2026-01-28,2026-10-15T16:20:02.150Z
8,3,6,1,5500,2026-01-13,2026-10-15T16:20:02.150Z
9,20,6,3,5500,2026-01-06,2026-10-15T16:20:02.150Z
10,18,12,4,450,2026-02-18,2026-10-15T16:20:02.150Z


In [0]:

from pyspark.sql import functions as F

sales_df = spark.table(
    "workspace.retail_training.bronze_sales"
)

sales_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("sale_id").alias("unique_sale_ids"),
    F.sum(
        F.when(F.col("sale_id").isNull(), 1).otherwise(0)
    ).alias("null_sale_ids"),
    F.sum(
        F.when(
            (F.col("quantity") <= 0) |
            (F.col("unit_price") < 0),
            1
        ).otherwise(0)
    ).alias("invalid_quantity_or_price"),
    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_ids")
).show()


+----------+---------------+-------------+-------------------------+-----------------+
|total_rows|unique_sale_ids|null_sale_ids|invalid_quantity_or_price|null_customer_ids|
+----------+---------------+-------------+-------------------------+-----------------+
|       100|            100|            0|                        0|                0|
+----------+---------------+-------------+-------------------------+-----------------+



In [0]:

from pyspark.sql import functions as F

schema = "workspace.retail_training"

# Read Bronze sales and dimension tables
sales = spark.table(f"{schema}.bronze_sales")
customers = spark.table(f"{schema}.bronze_customers")
products = spark.table(f"{schema}.bronze_products")
stores = spark.table(f"{schema}.bronze_stores")

# Normalize sales data types
sales_clean = (
    sales
    .withColumn("sale_id", F.col("sale_id").cast("int"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("store_id", F.col("store_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)"))
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn(
        "revenue",
        (F.col("quantity") * F.col("unit_price")).cast("decimal(14,2)")
    )
)

# Find invalid rows
invalid_condition = (
    F.col("sale_id").isNull()
    | F.col("customer_id").isNull()
    | F.col("product_id").isNull()
    | F.col("store_id").isNull()
    | F.col("quantity").isNull()
    | F.col("unit_price").isNull()
    | F.col("sale_date").isNull()
    | (F.col("quantity") <= 0)
    | (F.col("unit_price") < 0)
)

invalid_sales = sales_clean.filter(invalid_condition)

# Keep one record per sale_id for the initial training load
valid_sales = (
    sales_clean
    .filter(~invalid_condition)
    .dropDuplicates(["sale_id"])
)

# Validate dimension references
valid_sales = (
    valid_sales
    .join(customers.select("customer_id").distinct(),
          "customer_id", "left_semi")
    .join(products.select("product_id").distinct(),
          "product_id", "left_semi")
    .join(stores.select("store_id").distinct(),
          "store_id", "left_semi")
)

# Save clean sales as a Silver Delta table
(
    valid_sales
    .withColumn("_silver_processed_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{schema}.silver_sales")
)

print("Invalid sales before dimension checks:", invalid_sales.count())
print("Valid sales saved:", valid_sales.count())


Invalid sales before dimension checks: 0
Valid sales saved: 100


In [0]:
%sql

SELECT
    sale_id,
    customer_id,
    product_id,
    store_id,
    quantity,
    unit_price,
    revenue,
    sale_date
FROM workspace.retail_training.silver_sales
ORDER BY sale_id
LIMIT 10;


sale_id,customer_id,product_id,store_id,quantity,unit_price,revenue,sale_date
1,1,11,3,1,16000.00,16000.00,2026-02-01
2,4,4,5,2,9800.00,19600.00,2026-01-12
3,2,10,1,4,7200.00,28800.00,2026-01-12
4,17,4,5,2,9800.00,19600.00,2026-01-04
5,18,9,4,2,1800.00,3600.00,2026-01-29
6,1,8,2,3,28000.00,84000.00,2026-03-31
7,9,7,2,3,3200.00,9600.00,2026-01-28
8,3,6,4,1,5500.00,5500.00,2026-01-13
9,20,6,3,3,5500.00,16500.00,2026-01-06
10,18,12,1,4,450.00,1800.00,2026-02-18


In [0]:
%sql

SELECT
    COUNT(*) AS total_sales,
    COUNT(DISTINCT sale_id) AS unique_sales,
    SUM(quantity) AS total_units_sold,
    SUM(revenue) AS total_revenue,
    MIN(sale_date) AS first_sale_date,
    MAX(sale_date) AS last_sale_date
FROM workspace.retail_training.silver_sales;


total_sales,unique_sales,total_units_sold,total_revenue,first_sale_date,last_sale_date
100,100,254,2248000.00,2026-01-02,2026-03-31


In [0]:

from pyspark.sql import functions as F

schema = "workspace.retail_training"

sales = spark.table(f"{schema}.silver_sales")
products = spark.table(f"{schema}.bronze_products")
stores = spark.table(f"{schema}.bronze_stores")

# 1. Gold: Monthly sales summary
gold_monthly_sales = (
    sales
    .withColumn("sale_month", F.date_format("sale_date", "yyyy-MM"))
    .groupBy("sale_month")
    .agg(
        F.countDistinct("sale_id").alias("total_sales"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue")
    )
    .orderBy("sale_month")
)

# 2. Gold: Sales performance by product
gold_product_sales = (
    sales
    .groupBy("product_id")
    .agg(
        F.countDistinct("sale_id").alias("total_sales"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue")
    )
    .join(products.select("product_id", "product_name", "category"),
          "product_id", "left")
    .select(
        "product_id", "product_name", "category",
        "total_sales", "total_units_sold", "total_revenue"
    )
)

# 3. Gold: Sales performance by store
gold_store_sales = (
    sales
    .groupBy("store_id")
    .agg(
        F.countDistinct("sale_id").alias("total_sales"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("revenue").alias("total_revenue")
    )
    .join(stores.select("store_id", "store_name", "city"),
          "store_id", "left")
    .select(
        "store_id", "store_name", "city",
        "total_sales", "total_units_sold", "total_revenue"
    )
)

# Save Gold tables
for df, table_name in [
    (gold_monthly_sales, "gold_monthly_sales"),
    (gold_product_sales, "gold_product_sales"),
    (gold_store_sales, "gold_store_sales")
]:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{schema}.{table_name}")
    )
    print(f"Created {table_name}: {df.count()} rows")


Created gold_monthly_sales: 3 rows
Created gold_product_sales: 12 rows
Created gold_store_sales: 5 rows


In [0]:
%sql

SELECT *
FROM workspace.retail_training.gold_monthly_sales
ORDER BY sale_month;


sale_month,total_sales,total_units_sold,total_revenue
2026-01,38,102,986200.00
2026-02,28,64,461800.00
2026-03,34,88,800000.00


In [0]:
%sql
SELECT
    SUM(total_sales) AS total_sales,
    SUM(total_units_sold) AS total_units_sold,
    SUM(total_revenue) AS total_revenue
FROM workspace.retail_training.gold_monthly_sales;

total_sales,total_units_sold,total_revenue
100,254,2248000.00


In [0]:

batch_path = (
    "/Volumes/workspace/retail_training/raw_files/"
    "sales_batch_002.csv"
)

batch_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(batch_path)
)

print("New batch rows:", batch_df.count())
display(batch_df)


New batch rows: 5


sale_id,customer_id,product_id,store_id,quantity,unit_price,sale_date
101,1,1,2,2,2500,2026-04-01
102,2,2,3,3,1500,2026-04-02
103,3,3,4,4,3200,2026-04-03
104,4,4,5,1,9800,2026-04-04
105,5,5,1,2,450,2026-04-05


In [0]:

from pyspark.sql import functions as F

schema = "workspace.retail_training"

batch_path = (
    "/Volumes/workspace/retail_training/raw_files/"
    "sales_batch_002.csv"
)

batch_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(batch_path)
    .withColumn("_ingested_at", F.current_timestamp())
)

# Append this new batch to Bronze
(
    batch_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{schema}.bronze_sales")
)

print("Rows in new batch:", batch_df.count())
print(
    "Total Bronze sales:",
    spark.table(f"{schema}.bronze_sales").count()
)


Rows in new batch: 5
Total Bronze sales: 110


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

schema = "workspace.retail_training"

# Read only the new batch
source = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "/Volumes/workspace/retail_training/raw_files/"
        "sales_batch_002.csv"
    )
    .select(
        F.col("sale_id").cast("int"),
        F.col("customer_id").cast("int"),
        F.col("product_id").cast("int"),
        F.col("store_id").cast("int"),
        F.col("quantity").cast("int"),
        F.col("unit_price").cast("decimal(12,2)"),
        F.to_date("sale_date").alias("sale_date")
    )
    .withColumn(
        "revenue",
        (F.col("quantity") * F.col("unit_price")).cast("decimal(14,2)")
    )
)

# Basic validation
source = source.filter(
    F.col("sale_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("product_id").isNotNull()
    & F.col("store_id").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("unit_price").isNotNull()
    & (F.col("unit_price") >= 0)
    & F.col("sale_date").isNotNull()
)

# Keep only records with valid dimension references
customers = spark.table(f"{schema}.bronze_customers").select("customer_id").distinct()
products = spark.table(f"{schema}.bronze_products").select("product_id").distinct()
stores = spark.table(f"{schema}.bronze_stores").select("store_id").distinct()

source = (
    source
    .join(customers, "customer_id", "left_semi")
    .join(products, "product_id", "left_semi")
    .join(stores, "store_id", "left_semi")
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# Insert only previously unseen sale IDs
target = DeltaTable.forName(spark, f"{schema}.silver_sales")

(
    target.alias("t")
    .merge(source.alias("s"), "t.sale_id = s.sale_id")
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver sales total:", spark.table(f"{schema}.silver_sales").count())


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4908692925158174>, line 64
     57 # Insert only previously unseen sale IDs
     58 target = DeltaTable.forName(spark, f"{schema}.silver_sales")
     60 (
     61     target.alias("t")
     62     .merge(source.alias("s"), "t.sale_id = s.sale_id")
     63     .whenNotMatchedInsertAll()
---> 64     .execute()
     65 )
     67 print("Silver sales total:", spark.table(f"{schema}.silver_sales").count())

File /databricks/python/lib/python3.12/site-packages/delta/connect/tables.py:609, in DeltaMergeBuilder.execute(self)
    599 plan = MergeIntoTable(
    600     self._target,
    601     self._source,
   (...)    606     self._with_schema_evolution
    607 )
    608 df = DataFrame(plan, session=self._spark)
--> 609 return self._spark.createDataFrame(df.toPandas())

File /databricks/python/lib/python3.12/site-packages/pyspark/s

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

schema = "workspace.retail_training"

# Read the new records from Bronze, including ingestion metadata
source = (
    spark.table(f"{schema}.bronze_sales")
    .filter(F.col("sale_id").between(101, 105))
    .select(
        F.col("sale_id").cast("int"),
        F.col("customer_id").cast("int"),
        F.col("product_id").cast("int"),
        F.col("store_id").cast("int"),
        F.col("quantity").cast("int"),
        F.col("unit_price").cast("decimal(12,2)"),
        F.to_date("sale_date").alias("sale_date"),
        F.col("_ingested_at")
    )
    .withColumn(
        "revenue",
        (F.col("quantity") * F.col("unit_price")).cast("decimal(14,2)")
    )
)

# Validate required values
source = source.filter(
    F.col("sale_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("product_id").isNotNull()
    & F.col("store_id").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("unit_price").isNotNull()
    & (F.col("unit_price") >= 0)
    & F.col("sale_date").isNotNull()
)

# Validate dimension references
customers = spark.table(f"{schema}.bronze_customers").select("customer_id").distinct()
products = spark.table(f"{schema}.bronze_products").select("product_id").distinct()
stores = spark.table(f"{schema}.bronze_stores").select("store_id").distinct()

source = (
    source
    .join(customers, "customer_id", "left_semi")
    .join(products, "product_id", "left_semi")
    .join(stores, "store_id", "left_semi")
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# Merge only the new sale IDs
target = DeltaTable.forName(spark, f"{schema}.silver_sales")

(
    target.alias("t")
    .merge(source.alias("s"), "t.sale_id = s.sale_id")
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver sales total:", spark.table(f"{schema}.silver_sales").count())


Silver sales total: 110


In [0]:
%sql

SELECT
    sale_id,
    COUNT(*) AS record_count
FROM workspace.retail_training.silver_sales
GROUP BY sale_id
HAVING COUNT(*) > 1
ORDER BY sale_id;


sale_id,record_count
101,2
102,2
103,2
104,2
105,2


In [0]:
%sql

SELECT
    sale_id,
    COUNT(*) AS record_count
FROM workspace.retail_training.bronze_sales
WHERE sale_id BETWEEN 101 AND 105
GROUP BY sale_id
ORDER BY sale_id;


sale_id,record_count
101,2
102,2
103,2
104,2
105,2


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

schema = "workspace.retail_training"

sales = spark.table(f"{schema}.silver_sales")

# Keep one row per sale_id
window_spec = Window.partitionBy("sale_id").orderBy(
    F.col("_ingested_at").asc(),
    F.col("_silver_processed_at").asc()
)

deduplicated_sales = (
    sales
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

(
    deduplicated_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema}.silver_sales")
)

print("Silver rows after deduplication:",
      spark.table(f"{schema}.silver_sales").count())


Silver rows after deduplication: 105


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sale_id) AS unique_sale_ids
FROM workspace.retail_training.silver_sales;


total_rows,unique_sale_ids
105,105


In [0]:
%sql

SELECT
    sale_id,
    COUNT(*) AS record_count
FROM workspace.retail_training.bronze_sales
WHERE sale_id BETWEEN 101 AND 105
GROUP BY sale_id
ORDER BY sale_id;


sale_id,record_count
101,2
102,2
103,2
104,2
105,2


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

schema = "workspace.retail_training"
table_name = f"{schema}.bronze_sales"

bronze = spark.table(table_name)

window_spec = Window.partitionBy("sale_id").orderBy(
    F.col("_ingested_at").asc()
)

bronze_deduplicated = (
    bronze
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

(
    bronze_deduplicated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print("Bronze rows after deduplication:", spark.table(table_name).count())


Bronze rows after deduplication: 105


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sale_id) AS unique_sale_ids
FROM workspace.retail_training.bronze_sales;


total_rows,unique_sale_ids
105,105


In [0]:
%sql

SELECT
    'bronze_sales' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT sale_id) AS unique_sale_ids
FROM workspace.retail_training.bronze_sales

UNION ALL

SELECT
    'silver_sales',
    COUNT(*),
    COUNT(DISTINCT sale_id)
FROM workspace.retail_training.silver_sales;


table_name,row_count,unique_sale_ids
bronze_sales,105,105
silver_sales,105,105


In [0]:
%sql
SELECT 'bronze_customers' AS table_name, COUNT(*) AS row_count
FROM workspace.retail_training.bronze_customers

UNION ALL

SELECT 'bronze_products', COUNT(*)
FROM workspace.retail_training.bronze_products

UNION ALL

SELECT 'bronze_stores', COUNT(*)
FROM workspace.retail_training.bronze_stores

UNION ALL

SELECT 'bronze_sales', COUNT(*)
FROM workspace.retail_training.bronze_sales

UNION ALL

SELECT 'silver_sales', COUNT(*)
FROM workspace.retail_training.silver_sales

UNION ALL

SELECT 'gold_monthly_sales', COUNT(*)
FROM workspace.retail_training.gold_monthly_sales

UNION ALL

SELECT 'gold_product_sales', COUNT(*)
FROM workspace.retail_training.gold_product_sales

UNION ALL

SELECT 'gold_store_sales', COUNT(*)
FROM workspace.retail_training.gold_store_sales

ORDER BY table_name;

table_name,row_count
bronze_customers,20
bronze_products,12
bronze_sales,105
bronze_stores,5
gold_monthly_sales,4
gold_product_sales,12
gold_store_sales,5
silver_sales,105


In [0]:
%sql
SELECT
    COUNT(*) AS total_sales,
    COUNT(DISTINCT sale_id) AS unique_sale_ids,
    SUM(CASE WHEN sale_id IS NULL THEN 1 ELSE 0 END) AS null_sale_ids,
    MIN(sale_id) AS minimum_sale_id,
    MAX(sale_id) AS maximum_sale_id
FROM read_files(
    '/Volumes/workspace/retail_training/raw_files/sales_batch_002.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

total_sales,unique_sale_ids,null_sale_ids,minimum_sale_id,maximum_sale_id
5,5,0,101,105


In [0]:
%sql
WITH incoming_batch AS (
    SELECT DISTINCT sale_id
    FROM read_files(
        '/Volumes/workspace/retail_training/raw_files/sales_batch_002.csv',
        format => 'csv',
        header => true,
        inferSchema => true
    )
)
SELECT
    COUNT(*) AS batch_unique_ids,
    SUM(
        CASE WHEN b.sale_id IS NOT NULL THEN 1 ELSE 0 END
    ) AS ids_already_in_bronze,
    SUM(
        CASE WHEN b.sale_id IS NULL THEN 1 ELSE 0 END
    ) AS new_sale_ids
FROM incoming_batch i
LEFT JOIN workspace.retail_training.bronze_sales b
    ON i.sale_id = b.sale_id;

batch_unique_ids,ids_already_in_bronze,new_sale_ids
5,5,0


In [0]:
%sql
SELECT *
FROM workspace.retail_training.bronze_sales
LIMIT 5;

sale_id,customer_id,product_id,store_id,quantity,unit_price,sale_date,_ingested_at
1,1,11,3,1,16000,2026-02-01,2026-10-15T16:20:02.150Z
2,4,4,5,2,9800,2026-01-12,2026-10-15T16:20:02.150Z
3,2,10,1,4,7200,2026-01-12,2026-10-15T16:20:02.150Z
4,17,4,5,2,9800,2026-01-04,2026-10-15T16:20:02.150Z
5,18,9,4,2,1800,2026-01-29,2026-10-15T16:20:02.150Z


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sale_id) AS unique_sale_ids,
    SUM(CASE WHEN sale_id IS NULL THEN 1 ELSE 0 END) AS null_sale_ids,
    MIN(sale_id) AS minimum_sale_id,
    MAX(sale_id) AS maximum_sale_id
FROM read_files(
    '/Volumes/workspace/retail_training/raw_files/sales_batch_003.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

total_rows,unique_sale_ids,null_sale_ids,minimum_sale_id,maximum_sale_id
5,5,0,106,110


In [0]:
%sql
WITH batch AS (
    SELECT *
    FROM read_files(
        '/Volumes/workspace/retail_training/raw_files/sales_batch_003.csv',
        format => 'csv',
        header => true,
        inferSchema => true
    )
)
SELECT
    COUNT(*) AS batch_rows,
    SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END)
        AS missing_customers,
    SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END)
        AS missing_products,
    SUM(CASE WHEN s.store_id IS NULL THEN 1 ELSE 0 END)
        AS missing_stores
FROM batch b
LEFT JOIN workspace.retail_training.bronze_customers c
    ON b.customer_id = c.customer_id
LEFT JOIN workspace.retail_training.bronze_products p
    ON b.product_id = p.product_id
LEFT JOIN workspace.retail_training.bronze_stores s
    ON b.store_id = s.store_id;

batch_rows,missing_customers,missing_products,missing_stores
5,0,0,0


In [0]:
%sql
SELECT
    table_name,
    row_count,
    logged_at
FROM workspace.retail_training.pipeline_table_metrics
ORDER BY logged_at DESC, table_name
LIMIT 8;

table_name,row_count,logged_at
bronze_customers,20,2026-10-21T03:17:41.118Z
bronze_products,12,2026-10-21T03:17:41.118Z
bronze_sales,110,2026-10-21T03:17:41.118Z
bronze_stores,5,2026-10-21T03:17:41.118Z
gold_monthly_sales,4,2026-10-21T03:17:41.118Z
gold_product_sales,12,2026-10-21T03:17:41.118Z
gold_store_sales,5,2026-10-21T03:17:41.118Z
silver_sales,110,2026-10-21T03:17:41.118Z
